# SAAM Project — Part I, Section 2.2: Minimum-Variance Portfolio
**Group AT** — North America (AMER) / Scope 1+2

This notebook implements the out-of-sample minimum-variance portfolio:
1. Rolling 10-year estimation window (rebalanced annually)
2. Pairwise covariance estimation (NaN-robust) with PSD enforcement and diagonal loading
3. Long-only constrained optimization (weights ≥ 0, sum to 1)
4. Monthly ex-post performance with buy-and-hold weight drift
5. Sharpe ratio computed on monthly excess returns (Rp,t − Rf,t)

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ──────────────────────────────────────────────────────────
ESTIMATION_WINDOW = 120      # 10 years × 12 months
Y0                = 2013     # first allocation year
Y_END             = 2024     # last allocation year
DATA_DIR          = './cleaned_data/'
RIDGE_LAMBDA      = 1e-8     # diagonal loading for regularization
MIN_COMMON_OBS    = 24       # min shared months for pairwise cov entry

print('Setup complete.')

## 1. Load Cleaned Data

In [ ]:
import os
required_files = [
    'returns_monthly.csv', 'investment_sets.csv',
    'risk_free_rate.csv', 'firm_names.csv'
]
missing = [f for f in required_files
           if not os.path.exists(f'{DATA_DIR}{f}')]
if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}. Run data_cleaning.ipynb first "
        f"to generate cleaned data in {DATA_DIR}"
    )

# Monthly returns
returns_m = pd.read_csv(f'{DATA_DIR}returns_monthly.csv', index_col=0, parse_dates=False)
returns_m.columns = pd.to_datetime(returns_m.columns)

# Investment sets
inv_sets_df = pd.read_csv(f'{DATA_DIR}investment_sets.csv')
investment_sets = {int(Y): grp['ISIN'].tolist() for Y, grp in inv_sets_df.groupby('Year')}

# Risk-free rate
rf_df = pd.read_csv(f'{DATA_DIR}risk_free_rate.csv')
rf_dict = {(int(r['year']), int(r['month'])): r['RF_monthly']
           for _, r in rf_df.iterrows()}

# Firm names (as dict for fast lookup)
firm_names = pd.read_csv(f'{DATA_DIR}firm_names.csv', index_col=0).squeeze().to_dict()

# Return dates and December index mapping
dates_ret = returns_m.columns.tolist()
dec_ret_idx = {d.year: i for i, d in enumerate(dates_ret) if d.month == 12}

print(f'Returns: {returns_m.shape[0]} firms × {returns_m.shape[1]} months')
print(f'Investment sets: years {min(investment_sets.keys())}–{max(investment_sets.keys())}')
print(f'Risk-free rate: {len(rf_dict)} months')

## 2. Covariance Estimation

### Why not fillna(0)?
Filling NaN returns with 0 makes firms with sparse data appear artificially low-variance,
causing the optimizer to overweight them — exactly the firms with the worst data.

### Our approach:
1. **Pairwise covariance** via `pandas .cov(min_periods=...)`: for each pair (i,j),
   pandas internally demeans using only the months common to both i and j,
   then computes the covariance. This avoids the demeaning bias.
   We then adjust the denominator from (n−1) to n to match the project formula.
2. **PSD enforcement**: pairwise matrices can have negative eigenvalues.
   We eigendecompose and clip negatives to zero (nearest PSD matrix in Frobenius norm).
3. **Diagonal loading** (λI): ensures strict positive definiteness, even when N > T.

In [ ]:
def estimate_covariance_robust(ret_window, ridge_lambda=1e-8, min_common_obs=24):
    """
    Robust covariance estimation:
    1. Pairwise covariance via pandas (correct per-pair demeaning)
    2. Denominator adjustment: (n_ij - 1) → n_ij
    3. PSD enforcement (clip negative eigenvalues)
    4. Diagonal loading (ridge regularization)
    
    Parameters
    ----------
    ret_window : pd.DataFrame, shape (N_firms, T_months)
        Returns with potential NaN values. Firms in rows, months in columns.
    ridge_lambda : float
        Regularization parameter added to diagonal.
    min_common_obs : int
        Minimum common observations for a valid pairwise covariance entry.
    
    Returns
    -------
    cov_matrix : np.ndarray, shape (N, N)
        Regularized PSD covariance matrix.
    mu_hat : np.ndarray, shape (N,)
        Mean returns (per-firm, using each firm's available observations).
    n_neg : int
        Number of negative eigenvalues clipped during PSD enforcement.
    """
    N = ret_window.shape[0]
    
    # ── Step 1: Mean (per-firm, on available obs) ──────────────────────────
    # mu_hat is computed here for completeness but is not used by the min-variance
    # optimizer (which minimizes variance only, independently of expected returns).
    # It is returned for potential diagnostic use and discarded in the calling code.
    mu_hat = ret_window.mean(axis=1).values  # (N,)
    
    # ── Step 2: Pairwise covariance via pandas ─────────────────────────────
    # pandas .cov() transposes internally: expects observations in rows.
    # ret_window is (N_firms, T), so .T is (T, N_firms) → .cov() gives (N, N).
    # pandas uses ddof=1 (denominator n_ij - 1) and handles NaN pairwise:
    #   for each (i,j), it finds common non-NaN months, demeans using only
    #   those months, then computes the covariance. This is the correct approach.
    cov_pandas = ret_window.T.cov(min_periods=min_common_obs)  # (N, N)
    
    # ── Step 3: Adjust denominator from (n_ij - 1) to n_ij ─────────────────
    # METHODOLOGICAL NOTE (to report in the final paper):
    # The project formula prescribes 1/τ (τ=120). However, with pairwise
    # estimation, using a fixed τ for all pairs would distort variances
    # for pairs with fewer common observations. We use 1/n_ij (the number
    # of common valid observations for each pair) to preserve the geometric
    # properties of the covariance matrix in the presence of missing data.
    # pandas .cov() uses (n_ij - 1); we adjust to n_ij below.
    valid = (~ret_window.isna()).values.astype(float)  # (N, T)
    n_common = valid @ valid.T  # (N, N): number of common valid obs per pair
    
    # Adjustment factor: (n_ij - 1) / n_ij; avoid division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        adjust = np.where(n_common > 1, (n_common - 1) / n_common, 0.0)
    
    cov_matrix = cov_pandas.values * adjust
    
    # Replace any remaining NaN (from pairs with < min_common_obs) with 0
    cov_matrix = np.nan_to_num(cov_matrix, nan=0.0)
    
    # Ensure symmetry (numerical safety)
    cov_matrix = (cov_matrix + cov_matrix.T) / 2.0
    
    # ── Step 4: PSD enforcement ────────────────────────────────────────────
    eigvals, eigvecs = np.linalg.eigh(cov_matrix)
    n_neg = (eigvals < -1e-10).sum()
    eigvals_clipped = np.maximum(eigvals, 0.0)
    # Broadcasting — avoids allocating a dense diagonal matrix, ~2x faster
    cov_matrix = (eigvecs * eigvals_clipped) @ eigvecs.T
    cov_matrix = (cov_matrix + cov_matrix.T) / 2.0  # re-symmetrize
    
    # ── Step 5: Diagonal loading ───────────────────────────────────────────
    cov_matrix += ridge_lambda * np.eye(N)
    
    return cov_matrix, mu_hat, n_neg


print('Robust covariance estimator defined.')

## 3. Minimum-Variance Optimization

$$\min_{\alpha} \; \alpha' \Sigma_Y \alpha \quad \text{s.t.} \quad \alpha' \mathbf{1} = 1, \quad \alpha_i \geq 0 \; \forall i$$

In [ ]:
def solve_min_variance(cov_matrix, n_assets):
    """
    Solve the long-only minimum-variance portfolio.

    min  alpha' Sigma alpha
    s.t. sum(alpha) = 1
         alpha_i >= 0  for all i

    Returns weights, optimal variance, and convergence flag.
    """
    def objective(alpha):
        return alpha @ cov_matrix @ alpha

    def gradient(alpha):
        return 2.0 * cov_matrix @ alpha

    constraints = {'type': 'eq', 'fun': lambda a: np.sum(a) - 1.0}
    bounds = [(0.0, None)] * n_assets
    x0 = np.ones(n_assets) / n_assets

    result = minimize(
        objective, x0, jac=gradient, method='SLSQP',
        bounds=bounds, constraints=constraints,
        options={'maxiter': 1000, 'ftol': 1e-8}
    )

    if not result.success:
        print(f'  ⚠ Optimization warning: {result.message}')

    weights = result.x.copy()

    # Check for significant negative weights beyond SLSQP numerical noise
    if weights.min() < -1e-6:
        print(f'  ⚠ Significant negative weight detected: {weights.min():.2e}. '
              f'Results may be unreliable for this year.')

    # Clip small negatives (SLSQP numerical noise) and renormalize.
    # Note: renormalization slightly perturbs the optimal solution,
    # but the effect is negligible when clipped weights are near zero.
    weights = np.maximum(weights, 0.0)
    weights /= weights.sum()

    return weights, result.fun, result.success


print('Optimization function defined.')

## 4. Rolling Optimization: Year by Year

For each year $Y$ from 2013 to 2024:
1. Select investment set for year $Y$
2. Extract 120-month rolling window ending Dec $Y$
3. Estimate $\Sigma_Y$ (pairwise + PSD + ridge)
4. Solve min-variance
5. Roll forward by 1 year and repeat

In [ ]:
optimal_weights  = {}
estimation_stats = []

for Y in range(Y0, Y_END + 1):
    isins_Y  = investment_sets[Y]
    n_assets = len(isins_Y)

    # Rolling window: 120 months ending Dec Y
    end_idx   = dec_ret_idx[Y]
    start_idx = end_idx - ESTIMATION_WINDOW + 1
    window_cols = dates_ret[start_idx : end_idx + 1]

    # Guard: ensure full estimation window is available
    assert len(window_cols) == ESTIMATION_WINDOW, (
        f"Year {Y}: window has {len(window_cols)} months, "
        f"expected {ESTIMATION_WINDOW}."
    )

    ret_window = returns_m.loc[isins_Y, window_cols]

    # mu_hat discarded (_): min-variance optimization does not use expected returns
    cov_matrix, _, n_neg_eig = estimate_covariance_robust(
        ret_window, ridge_lambda=RIDGE_LAMBDA, min_common_obs=MIN_COMMON_OBS
    )

    weights, opt_var, opt_success = solve_min_variance(cov_matrix, n_assets)
    optimal_weights[Y] = pd.Series(weights, index=isins_Y)

    n_nonzero        = (weights > 1e-8).sum()
    top_weight       = weights.max()
    # ann_vol_insample: in-sample annualized volatility from the optimizer.
    # This is NOT the ex-post volatility (computed later in Cell 6).
    ann_vol_insample = np.sqrt(opt_var * 12)

    estimation_stats.append({
        'Year'              : Y,
        'N_firms'           : n_assets,
        'N_nonzero_wts'     : n_nonzero,
        'Max_weight'        : top_weight,
        'Min_var_monthly'   : opt_var,
        'Ann_vol_insample'  : ann_vol_insample,
        'Neg_eigvals_clipped': n_neg_eig,
        'Opt_success'       : opt_success,
    })

    print(f'  {Y}: {n_assets} firms, {n_nonzero} non-zero wts, '
          f'max wt = {top_weight:.3f}, '
          f'ann vol (in-sample) = {ann_vol_insample:.4f}, '
          f'neg eigvals = {n_neg_eig}, '
          f'converged = {opt_success}')

df_stats = pd.DataFrame(estimation_stats).set_index('Year')

# Report years where the optimizer did not converge
n_failed = sum(1 for s in estimation_stats if not s['Opt_success'])
if n_failed > 0:
    print(f'\n⚠ Optimization failed to converge for '
          f'{n_failed}/{Y_END - Y0 + 1} years.')
else:
    print('\n✓ Optimization converged for all years.')

print('\nDone. Optimal weights computed for all years.')

## 5. Ex-Post Portfolio Returns (Buy-and-Hold Drift)

$$R_{p,t+k} = \alpha_{t+k-1}' R_{t+k}$$

$$\alpha_{i,t+k-1} = \frac{\alpha_{i,t+k-2} \times (1 + R_{i,t+k-1})}{1 + R_{p,t+k-1}}$$

In [ ]:
portfolio_returns = []

for Y in range(Y0, Y_END + 1):
    w_Y      = optimal_weights[Y]
    isins_Y  = w_Y.index.tolist()
    year_next = Y + 1

    # Restrict strictly to Jan–Dec of year_next.
    # PDF specifies T=144 months (Jan 2014–Dec 2025). Jan 2026, which is present
    # in the raw return data, must be excluded.
    months_next = [d for d in dates_ret
                   if d.year == year_next and d <= pd.Timestamp('2025-12-31')]

    if not months_next:
        continue

    # Costless instantaneous rebalancing assumed at each year-end.
    # Transaction costs are not modeled (limitation to note in the report).
    alpha_current = w_Y.values.copy()

    for dt in months_next:
        # NaN returns set to 0: covers delisted firms (after -100% in data_cleaning)
        # and data gaps (treated conservatively as no-return months).
        r_stocks = returns_m.loc[isins_Y, dt].fillna(0.0).values
        r_port   = alpha_current @ r_stocks
        portfolio_returns.append({'Date': dt, 'Return': r_port})

        # Buy-and-hold weight drift within the year
        if abs(1 + r_port) > 1e-12:
            alpha_current = alpha_current * (1 + r_stocks) / (1 + r_port)

df_port_ret = pd.DataFrame(portfolio_returns).set_index('Date').sort_index()

# Verify sample length matches PDF specification
assert len(df_port_ret) == 144, (
    f"Expected 144 months (Jan 2014–Dec 2025), got {len(df_port_ret)}. "
    f"Last date in series: {df_port_ret.index[-1].strftime('%Y-%m')}. "
    f"Check whether Jan 2026 was incorrectly included."
)
print(f'Portfolio return series: {len(df_port_ret)} months ✓')
print(f'Period: {df_port_ret.index[0].strftime("%Y-%m")} to '
      f'{df_port_ret.index[-1].strftime("%Y-%m")}')
df_port_ret.head()

## 6. Portfolio Performance Statistics

### Sharpe Ratio on excess returns:
$$ER_{p,t} = R_{p,t} - R_{f,t}$$
$$SR_p = \frac{\overline{ER}_p \times 12}{\sigma_p \times \sqrt{12}}$$

We annualize the mean excess return (×12) and divide by the annualized
portfolio volatility ($\sigma_p$), as per the standard academic definition.

In [ ]:
r_p = df_port_ret['Return'].values

# ── Excess returns: Rp,t - Rf,t (month by month) ──────────────────────────
rf_series = np.array([rf_dict.get((dt.year, dt.month), 0.0) 
                       for dt in df_port_ret.index])
excess_returns = r_p - rf_series

# ── Raw return statistics ──────────────────────────────────────────────────
mu_ann  = np.mean(r_p) * 12
vol_ann = np.std(r_p, ddof=0) * np.sqrt(12)

# ── Sharpe: (annualized mean excess return) / (annualized portfolio vol) ──
er_mean_ann = np.mean(excess_returns) * 12
# Sharpe ratio: annualized mean excess return divided by annualized total-return
# volatility. We use std(r_p) rather than std(excess_returns) in the denominator,
# which is standard practice when the risk-free rate is treated as non-stochastic.
# The difference is negligible given the low and stable risk-free rate in this sample.
sharpe = er_mean_ann / vol_ann

# ── Reference ──────────────────────────────────────────────────────────────
rf_ann = np.mean(rf_series) * 12
r_min = np.min(r_p)
r_max = np.max(r_p)

print('═══════════════════════════════════════════════════════')
print('  MINIMUM-VARIANCE PORTFOLIO — P(mv)_oos')
print('═══════════════════════════════════════════════════════')
print(f'  Annualized return (μ̄_p):     {mu_ann:>8.4f}  ({mu_ann*100:.2f}%)')
print(f'  Annualized volatility (σ_p):  {vol_ann:>8.4f}  ({vol_ann*100:.2f}%)')
print(f'  Risk-free rate (avg ann.):     {rf_ann:>8.4f}  ({rf_ann*100:.2f}%)')
print(f'  Sharpe ratio (excess ret):     {sharpe:>8.4f}')
print(f'  Min monthly return:            {r_min:>8.4f}  ({r_min*100:.2f}%)')
print(f'  Max monthly return:            {r_max:>8.4f}  ({r_max*100:.2f}%)')
print('═══════════════════════════════════════════════════════')

## 7. Cumulative Return Series

In [ ]:
CHARTS_DIR = './Charts/'
os.makedirs(CHARTS_DIR, exist_ok=True)

cumulative_ret = (1 + df_port_ret['Return']).cumprod()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cumulative_ret.index, cumulative_ret.values,
        color='#1f4e79', linewidth=1.5, label='P(mv)_oos')
ax.set_title('Cumulative Return — Minimum-Variance Portfolio (Long-Only)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Wealth Index (starting at 1)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}mv_cumulative_return.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Final wealth: {cumulative_ret.iloc[-1]:.4f} '
      f'(total return: {(cumulative_ret.iloc[-1]-1)*100:.2f}%)')

## 8. Weight Analysis

In [ ]:
for Y in [Y0, Y_END]:
    w = optimal_weights[Y]
    w_sorted = w[w > 1e-6].sort_values(ascending=False)
    top10 = w_sorted.head(10)
    
    print(f'\nTop 10 holdings at end of {Y} (for {Y+1}):')
    print(f'{"ISIN":<20} {"Name":<35} {"Weight":>8}')
    print('-' * 65)
    for isin, weight in top10.items():
        name = str(firm_names.get(isin, 'N/A'))[:33]
        print(f'{isin:<20} {name:<35} {weight:>8.4f}')
    print(f'Sum of top 10: {top10.sum():.4f}')
    print(f'Total non-zero weights: {(w > 1e-6).sum()}')

## 9. Estimation Diagnostics

In [ ]:
print('Optimization summary across years:\n')
print(df_stats.round(6).to_string())

## 10. Save Results

In [ ]:
# Portfolio returns
df_port_ret.to_csv(f'{DATA_DIR}mv_portfolio_returns.csv')

# Optimal weights (long format, non-zero only)
weights_rows = []
for Y, w in optimal_weights.items():
    for isin, wt in w.items():
        if wt > 1e-10:
            weights_rows.append({'Year': Y, 'ISIN': isin, 'Weight': wt})
pd.DataFrame(weights_rows).to_csv(f'{DATA_DIR}mv_optimal_weights.csv', index=False)

# Summary statistics
pd.DataFrame([{
    'Portfolio': 'P(mv)_oos',
    'Ann_Return': mu_ann,
    'Ann_Volatility': vol_ann,
    'Sharpe_Ratio': sharpe,
    'Min_Monthly_Return': r_min,
    'Max_Monthly_Return': r_max,
    'RF_ann': rf_ann,
}]).to_csv(f'{DATA_DIR}mv_summary_stats.csv', index=False)

# The diagnostics DataFrame now contains 'Ann_vol_insample' and 'Opt_success'
# instead of the old 'Ann_vol'. No other changes to the save block are needed.
df_stats.to_csv(f'{DATA_DIR}mv_estimation_diagnostics.csv')

print('All results saved:')
print(f'  - {DATA_DIR}mv_portfolio_returns.csv')
print(f'  - {DATA_DIR}mv_optimal_weights.csv')
print(f'  - {DATA_DIR}mv_summary_stats.csv')
print(f'  - {DATA_DIR}mv_estimation_diagnostics.csv')

print('\nMethodological notes for the report:')
print('  - Covariance set to 0 for pairs with fewer than MIN_COMMON_OBS common '
      'observations: implies independence assumption for sparse pairs.')
print('  - Diagonal loading lambda=1e-8 ensures strict PD for the solver; '
      'negligible relative to smallest non-zero eigenvalue (~1e-4).')
print('  - No transaction costs modeled at annual rebalancing.')
print('  - revenue_mln_usd.csv is already in million USD — do NOT divide by 1000 in Part II.')